# Setup

In [ ]:
%run notebook_setup.py

import pandas as pd
import numpy as np
from scipy.stats import trim_mean
import polars as pl
import pandas as pd
import logging
import gc

import matplotlib.pyplot as plt
import seaborn as sns

from src.config.dir_config import OUTPUT_PATH_DEMAND_SUMMARY
from src.config.bigquery_config import CREDENTIALS_GBQ, PROJECT_ID_GBQ
from src.utils.read_data import read_data
from src.utils.setup_logging import setup_logging
from src.utils.create_week_date import add_week_start_date
from src.utils.export_as_excel import export_dataframes_as_tables


import logging

setup_logging()

# Data import

In [ ]:
data_genex = pd.read_parquet('../data/processed/genex_data_processed.parquet')
demand_summary = pd.read_parquet('../data/processed/demand_summary.parquet')
data_sales = pd.read_parquet('../data/processed/weekly_sales_by_season/Invierno/weekly_sales_Invierno_2025_processed.parquet')
classifications = pd.read_excel('../data/external/Consolidado clasificaciones modelo BI.xlsx')
transfers = pd.read_parquet('../data/processed/trf_data_processed.parquet')

# Data processing

In [ ]:
data_sales = data_sales[data_sales['cod_sucursal'] != 767]
demand_summary = demand_summary[demand_summary['cod_sucursal'] != 767]

In [ ]:
demand_info = demand_summary[['cod_producto', 'cod_talla', 'cod_sucursal',
                'nombre_sucursal',
                'nombre_temporada','ano_temporada','nombre_depto','nombre_linea','nom_talla',
                'demand_type']].copy()

In [ ]:
transfers_pivot = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla','cod_ano_comercial','cod_semana'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot.columns.name = None

In [ ]:
transfers_pivot_summary = transfers.pivot_table(
    index=['cod_sucursal','cod_producto','cod_talla'],
    observed=True,
    columns="nombre_razon_group",
    values= 'cantidad_des',
    aggfunc="sum",
    fill_value=0
).reset_index()

transfers_pivot_summary.columns.name = None

In [ ]:
logging.info('Adding dimensions to data_all')

logging.info('Merging genex data with data_all')
data_all = data_sales.merge(data_genex,
                            on=['cod_producto','cod_talla', 'cod_sucursal',
                                'cod_ano_comercial','cod_semana'],
                                how='left')

logging.info('Merging demand info with data_all')
data_all = data_all.merge(demand_info,
                            on=['cod_producto','cod_talla', 'cod_sucursal'],
                            how='left')

logging.info('Merging classifications with data_all')
data_all = data_all.merge(classifications,
                            on=['nombre_temporada','cod_sucursal','nombre_depto', 'nombre_linea'],
                            how='left')

logging.info("Merging with transfers with data_all")
data_all = data_all.merge(transfers_pivot,
                            on=['cod_producto','cod_talla', 'cod_sucursal',
                                'cod_ano_comercial','cod_semana'],
                                how='left')

logging.info('Deleting intermediate dataframes to free memory')
del demand_info, classifications, data_sales
gc.collect()

In [ ]:
data_all['clasificacion'] = pd.Categorical(data_all['clasificacion'], categories=['AA','A','B','C'], ordered=True)

data_all = add_week_start_date(data_all)

data_all['factor_l_dias'] = np.where(
    data_all['mean_sales_past_4_weeks'] == 0,
    np.nan,
    (data_all['vta_promedio'] / data_all['mean_sales_past_4_weeks']).round(3)
)

data_all[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = data_all[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

In [ ]:
demand_summary = demand_summary.merge(
    transfers_pivot_summary,
    on = ['cod_sucursal','cod_producto','cod_talla'],
    how = 'left'
)

demand_summary[["PREDISTRIBUIDA",
        "REPOSICION AUTOMATIC",
        "CARGA MANUAL",
        "OTRAS"]] = demand_summary[["PREDISTRIBUIDA",
                                    "REPOSICION AUTOMATIC",
                                    "CARGA MANUAL",
                                    "OTRAS"]].fillna(0)

# EDA

## Sample

In [ ]:
depto = "Juvenil mujer"
linea = "Lanas"
cod_producto = 671445

data_sample = data_all[(data_all['nombre_depto'] == depto) & (data_all['nombre_linea'] == linea)].copy()
data_sample = data_sample[data_sample['cod_producto'] == cod_producto]

interest_columns = ['cod_sucursal','nombre_sucursal',
                    'nombre_depto','nombre_linea','cod_producto', 'cod_talla', 'nom_talla',
                    'date','mnt_precio_vigente','week_number','stock_end_week','weekly_sales',
                    'mean_sales_past_4_weeks','mean_sales_next_4_weeks','vta_promedio',
                    'factor','semana_vta','repo_x_dda','can_final','clasif',"PREDISTRIBUIDA","REPOSICION AUTOMATIC","CARGA MANUAL"]


data_sample = data_sample[interest_columns].round(2).reset_index(drop=True)
demand_summary_sample = demand_summary[demand_summary['cod_producto'] == cod_producto].copy()

In [ ]:
data_sample.query('cod_sucursal == 32')

In [ ]:
data_sample.groupby(['cod_producto','cod_sucursal','cod_talla','week_number'])[['REPOSICION AUTOMATIC']].sum().reset_index().query('week_number == 2').nlargest(10,'REPOSICION AUTOMATIC')

In [ ]:
dataframes = {
    "sales_by_week": data_sample,
    "demand_summary":demand_summary_sample
    }

export_dataframes_as_tables(
    dataframes,
    '../sandbox/sample_repo_sin_estadistica.xlsx'
)

# Ays 1

In [ ]:
data_all.columns

In [ ]:
week_number_repo = (
   data_all.groupby(['week_number'], observed=True)
   .agg(
       repo_auto = ('REPOSICION AUTOMATIC', 'sum'),
       carga_manual = ('CARGA MANUAL', 'sum'),
       otras = ('OTRAS', 'sum')
       )
   .reset_index()
)

# Ajuste de semana si 0 es llegada
week_number_repo['week_number'] = week_number_repo['week_number'] - 1

week_number_repo = week_number_repo.melt(
    id_vars='week_number',
    value_vars=['repo_auto', 'carga_manual', 'otras'],
    var_name='tipo_reposicion',
    value_name='cantidad'
)

week_number_repo['week_number_group'] = week_number_repo['week_number'].apply(
    lambda x: str(x) if x < 15 else '15+'
)

week_number_repo['cantidad_prop'] = week_number_repo['cantidad'] / week_number_repo['cantidad'].sum()

week_number_repo.sort_values('cantidad_prop', ascending=False)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Agrupar datos por grupo de semana y tipo de reposición
grouped_plot_data = (
    week_number_repo
    .groupby(['week_number_group', 'tipo_reposicion'], observed=True)
    .agg(cantidad=('cantidad', 'sum'))
    .reset_index()
)

# Calcular proporción dentro de cada semana
grouped_plot_data['cantidad_prop'] = grouped_plot_data['cantidad'] / grouped_plot_data['cantidad'].sum()

# Ordenar semanas correctamente
orden_semanas = [str(i) for i in range(0, 15)] + ['15+']
grouped_plot_data['week_number_group'] = pd.Categorical(
    grouped_plot_data['week_number_group'],
    categories=orden_semanas,
    ordered=True
)

# Graficar
plt.figure(figsize=(10, 5))
sns.barplot(
    data=grouped_plot_data,
    x='week_number_group',
    y='cantidad_prop',
    hue='tipo_reposicion',
    hue_order=['repo_auto', 'carga_manual', 'otras']
)

plt.title('Proporción de tipo de reposición por semana')
plt.ylabel('Proporción')
plt.xlabel('Semanas desde llegada a sucursal')
plt.legend(title='Tipo de reposición')
plt.tight_layout()
plt.show()

## Por depto

In [ ]:
data_all.columns

In [ ]:
repo_summary =(
   data_all.groupby(['nombre_depto','nombre_linea','week_number'], observed=True)
   .agg(
       repo_auto = ('REPOSICION AUTOMATIC', 'sum'),
       carga_manual = ('CARGA MANUAL', 'sum'),
       otras = ('OTRAS', 'sum')
       )
   .reset_index()
)

# Ajuste de semana si 0 es llegada
repo_summary['week_number'] = repo_summary['week_number'] - 1

repo_summary = repo_summary.melt(
    id_vars=['nombre_depto','nombre_linea','week_number'],
    value_vars=['repo_auto', 'carga_manual', 'otras'],
    var_name='tipo_reposicion',
    value_name='cantidad'
)


repo_summary['cantidad_prop'] = repo_summary['cantidad'] / repo_summary.groupby(['nombre_depto', 'nombre_linea'])['cantidad'].transform('sum')

repo_summary['cantidad_prop'] = repo_summary['cantidad_prop'].fillna(0)

In [ ]:
repo_summary.query("nombre_depto == 'Juvenil mujer' and nombre_linea == 'Lanas'").sort_values('cantidad_prop', ascending = False).head(15)

In [ ]:
repo_summary.to_excel('../sandbox/repo_summary.xlsx', index=False)